# JINJA Processor

When working with this, one should be extremely cautious not to remove unnecessary components 

Redshift supports some configs, which are not supported by Trino. Therefore, these keyword arguments should be removed:  
- `sort` 
- `dist`

Additionally, based on the previous analysis, there is a handful of other keys that are used that are meaningless: 
- `unique_id`
- `distkey`
- `tag`
- `materialize`
- `dist_key`
- `sort_key`
- `sortkey`

---

`staging` layer will be converted into the views, therefore need to remove everything associated with incremental builds: 
- `materialized` - this will be declared in `profiles.yml`
- `is_incremental()` condition 
- `on_schema_change` - specific config for incremental 
Notes: 
- `unique_key` can stay

---

### Imports

In [1]:
import os
import glob
import re
import traceback
import jinja2
from typing import List, Optional, Any
from jinja2 import Environment, BaseLoader

---

### JINJA Processor 

Creating a JINJA processor offers flexibility, reusability, and maintainability while providing powerful features for template rendering and simulation of dbt macros. It can be leveraged with dynamic cleaning of SQL snippets.

In [67]:
class DbtJinjaProcessor:
    """A specialized Jinja processor that simulates a subset of dbt's Jinja macros.

    This processor:
    - Preserves Jinja code for `source`, `ref`, `dbt_utils` macros as literal strings.
    - Controls how incremental logic is handled via `is_incremental`. If True, incremental blocks remain as-is;
      if False, incremental blocks are omitted entirely.
    - Excludes certain arguments from `{{ config(...) }}` calls. By default, all arguments are kept unless
      explicitly listed in `excluded_config_args`. If after exclusion no arguments remain, the config block
      is removed.
    """
    # Adding a list of mock functions that will just return the mock function later
    mock_function_names = [
        "fico_extract",
        "date_formatting",
        "cxo_business_mins_between__2",
        "cxo_call_metric_outliers",
        "target"
    ]
    
    def __init__(self, 
                 excluded_config_args: Optional[List[str]] = None, 
                 add_config_args: Optional[dict] = None,
                 is_incremental: bool = False,
                 force_config_if_missing: bool = True
                ) -> None:
        """
        Initialize the DbtJinjaProcessor.

        Args:
            excluded_config_args:
                A list of argument names to exclude from config() calls.
                For example, ['materialized', 'strategy'] will remove these arguments from any config blocks.
                All other arguments remain intact. If no arguments remain after exclusion, the config block is removed.
            
            add_config_args:
                A dictionary of arguments to add to the config() calls (e.g. {"materialized": "view"}).
                
            is_incremental:
                If True, {% if is_incremental() %} blocks are rendered as-is.
                If False, these blocks are omitted from the output.

            force_config_if_missing:
                If True, a default config call is prepended to the model if no config call exists at all.
                
        """
        self.excluded_config_args = excluded_config_args or []
        self.add_config_args = add_config_args or {}
        self.is_incremental = is_incremental
        self.force_config_if_missing = force_config_if_missing
        
        self.env = Environment(loader=BaseLoader())
        self._set_globals()

    def _set_globals(self) -> None:
        """Set the global functions in the Jinja environment to mock dbt macros."""
        self.env.globals['source'] = self._mock_source
        self.env.globals['ref'] = self._mock_ref
        self.env.globals['is_incremental'] = lambda: self.is_incremental
        self.env.globals['config'] = self._mock_config
        self.env.globals['dbt_utils'] = self._get_mock_dbt_utils()
        self.env.globals['var'] = self._mock_var
        self.env.globals['finance_schema'] = self._mock_finance_schema
        self.env.globals['mtn_time_convert'] = self._mock_mtn_time_convert
        self.env.globals['dbt'] = self._get_mock_dbt()
        self.env.globals['divvy_whitelabel_programs'] = self._mock_divvy_whitelabel_programs
        self._generate_mock_functions(self.env, self.mock_function_names)
    
    def _generate_mock_functions(self, env, function_names):
        for name in function_names:
            def mock_function(*args, name=name):
                arg_strings = [repr(arg) for arg in args]
                all_args = ', '.join(arg_strings)
                return f"{{{{ {name}({all_args}) }}}}"
            env.globals[name] = mock_function  
            
    def _mock_source(self, schema: str, table: str) -> str:
        """Return the literal Jinja code for source."""
        return f"{{{{ source('{schema}', '{table}') }}}}"

    def _mock_ref(self, model_name: str) -> str:
        """Return the literal Jinja code for ref."""
        return f"{{{{ ref('{model_name}') }}}}"

    def _mock_config(self, *args: Any, **kwargs: Any) -> str:
        """
        Only rewrite config() if something needs to be excluded or added.
        Otherwise, return a placeholder to keep the original unchanged.
        """
        original_config_marker = "__DBT_ORIGINAL_CONFIG__"
    
        # If no exclusions or additions, return a placeholder so we don't overwrite
        if not self.excluded_config_args and not self.add_config_args:
            return original_config_marker
    
        # Else: apply exclusions and additions
        filtered_kwargs = {k: v for k, v in kwargs.items() if k not in self.excluded_config_args}
        filtered_kwargs.update(self.add_config_args)
    
        if not filtered_kwargs:
            return ""
    
        arg_strings = []
        for k, v in filtered_kwargs.items():
            val_str = "[" + ", ".join(repr(item) for item in v) + "]" if isinstance(v, list) else repr(v)
            arg_strings.append(f"{k}={val_str}")
    
        return f"{{{{ config({', '.join(arg_strings)}) }}}}"


    def _mock_dbt_utils(self, macro: str, *args: Any, **kwargs: Any) -> str:
        """
        Mock any dbt_utils macro.

        Args:
            macro (str): The name of the dbt_utils macro to be mocked.
            *args (Any): Positional arguments to be passed to the macro.
            **kwargs (Any): Keyword arguments to be passed to the macro.

        Returns:
            str: A string containing the Jinja template code for the macro call.
        """
        # Construct the original macro call with arguments
        arg_strings = [repr(arg) for arg in args]  # Positional arguments
        kwarg_strings = [f"{k}={repr(v)}" for k, v in kwargs.items()]  # Keyword arguments

        # Combine positional and keyword arguments
        all_args = ', '.join(arg_strings + kwarg_strings)

        return f"{{{{ dbt_utils.{macro}({all_args}) }}}}"

    def _get_mock_dbt_utils(self):
        """Return a mock dbt_utils object with dynamic macro handling."""
        class MockDbtUtils:
            def __init__(self, mock_func):
                self._mock_func = mock_func

            def __getattr__(self, name):
                return lambda *args, **kwargs: self._mock_func(name, *args, **kwargs)

        return MockDbtUtils(self._mock_dbt_utils)

    def _mock_dbt_method(self, method: str, *args: Any, **kwargs: Any) -> str:
        """Mock any dbt method."""
        arg_strings = [repr(arg) for arg in args]
        kwarg_strings = [f"{k}={repr(v)}" for k, v in kwargs.items()]
        all_args = ', '.join(arg_strings + kwarg_strings)
        return f"{{{{ dbt.{method}({all_args}) }}}}"
    
    def _get_mock_dbt(self):
        """Return a mock dbt object with dynamic method handling."""
        class MockDbt:
            def __init__(self, mock_func):
                self._mock_func = mock_func
    
            def __getattr__(self, name):
                return lambda *args, **kwargs: self._mock_func(name, *args, **kwargs)
    
        return MockDbt(self._mock_dbt_method)
            
    def _mock_var(self, var_name: str, default: Any = None) -> str:
        """
        Mock the dbt var function.
        
        Args:
            var_name (str): The name of the variable to be retrieved.
            default (Any): The default value to return if the variable is not found.

        Returns:
            str: A string containing the Jinja template code for the var call.
        """
        if default is not None:
            return f"{{{{ var('{var_name}', {repr(default)}) }}}}"
        return f"{{{{ var('{var_name}') }}}}"
    
    def _mock_finance_schema(self, schema_name: str) -> str:
        """Return the literal Jinja code for finance_schema."""
        return f"{{{{ finance_schema('{schema_name}') }}}}"

    def _mock_mtn_time_convert(self, field_name: str) -> str:
        """Return the literal Jinja code for mtn_time_convert."""
        return f"{{{{ mtn_time_convert('{field_name}') }}}}"
    
    def _mock_divvy_whitelabel_programs(self) -> str:
        return "{{ divvy_whitelabel_programs() }}"   

    def process_template_str(self, template_str: str) -> str:
        has_config = "{{ config(" in template_str
        has_incremental = "{% if is_incremental()" in template_str
        modifying_config = bool(self.excluded_config_args or self.add_config_args)
        should_force_config = self.force_config_if_missing and self.add_config_args and not has_config
        should_remove_incremental = not self.is_incremental and has_incremental
    
        # ✅ Skip entirely if nothing changes
        if not modifying_config and not should_force_config and not should_remove_incremental:
            return template_str
    
        # Inject config if needed
        if should_force_config:
            config_parts = []
            for k, v in self.add_config_args.items():
                val_str = "[" + ", ".join(repr(item) for item in v) + "]" if isinstance(v, list) else repr(v)
                config_parts.append(f"{k}={val_str}")
            config_str = f"{{{{ config({', '.join(config_parts)}) }}}}\n"
            template_str = config_str + template_str
    
        # Remove incremental blocks if needed
        if should_remove_incremental:
            template_str = re.sub(
                r"{%\s*if is_incremental\(\)\s*%}.*?{%\s*endif\s*%}",
                "",
                template_str,
                flags=re.DOTALL
            )
    
        # Render
        rendered = self.env.from_string(template_str).render()
    
        # Restore untouched config blocks
        rendered = rendered.replace("__DBT_ORIGINAL_CONFIG__", "{{ config(...) }}")
    
        # Strip extra blank lines
        cleaned = "\n".join(line for line in rendered.splitlines() if line.strip())
        return cleaned

In [68]:
test_query_1 = """

    {{ config(
        materialized='incremental',
        unique_key='id',
        strategy='merge',
        sort=['organizationid'],
        dist='organizationid',
        tags=['datalake_sync']
    ) }}
    
    SELECT
        id,
        case when solved_at_pst is not null then {{ cxo_business_mins_between__2('created_at_pst', 'solved_at_pst') }} * 1.0, 
        {{date_formatting('date_five9')}}, 
        {{ dbt_utils.generate_surrogate_key(['vigilancedecisionid', 'createddate']) }} AS unique_id,
        {{ dbt_utils.generate_surrogate_key('id', 'timestamp') }} AS other_unique id, 
        {{fico_extract('fico_range')}} as vantage_score,
        isactive,
        cast(activity_date as {{ dbt.type_timestamp() }}) as activity_timestamp,
        name,
        createddate,
        createdby,
        updateddate,
        updatedby,
        addresscity,
        orgvcardstatus,
        vendorvcardstatus,
        orgvcardoptoutreason,
        vstatusupdatesource,
        intlpaymenttype,
        processorvcardstatus,
        ,{{ mtn_time_convert('_fivetran_synced') }}  ladda
    FROM {{ source('bdc_nopii', 'vendor') }}
    WHERE
    bank_program_uuid NOT IN {{ divvy_whitelabel_programs() }}
    {% if is_incremental() %}
    AND updateddate >= date_add('day', -2, current_date)
        
    {% endif %}
    
                """

In [69]:
processor = DbtJinjaProcessor(excluded_config_args=["materialized", "strategy", "dist", "sort", "sortkey"],
                              add_config_args={"materialized": "view"},
                              is_incremental=True)
result = processor.process_template_str(test_query_1)
print(result)

    {{ config(unique_key='id', tags=['datalake_sync'], materialized='view') }}
    SELECT
        id,
        case when solved_at_pst is not null then {{ cxo_business_mins_between__2('created_at_pst', 'solved_at_pst') }} * 1.0, 
        {{ date_formatting('date_five9') }}, 
        {{ dbt_utils.generate_surrogate_key(['vigilancedecisionid', 'createddate']) }} AS unique_id,
        {{ dbt_utils.generate_surrogate_key('id', 'timestamp') }} AS other_unique id, 
        {{ fico_extract('fico_range') }} as vantage_score,
        isactive,
        cast(activity_date as {{ dbt.type_timestamp() }}) as activity_timestamp,
        name,
        createddate,
        createdby,
        updateddate,
        updatedby,
        addresscity,
        orgvcardstatus,
        vendorvcardstatus,
        orgvcardoptoutreason,
        vstatusupdatesource,
        intlpaymenttype,
        processorvcardstatus,
        ,{{ mtn_time_convert('_fivetran_synced') }}  ladda
    FROM {{ source('bdc_nopii', 'vendor'

In [70]:
# If we run with is_incremental=True, incremental blocks remain
processor_incremental = DbtJinjaProcessor(excluded_config_args=["materialized"], is_incremental=True)
result_incremental = processor_incremental.process_template_str(test_query_1)
print("\nWith is_incremental=True:\n", result_incremental)


With is_incremental=True:
     {{ config(unique_key='id', strategy='merge', sort=['organizationid'], dist='organizationid', tags=['datalake_sync']) }}
    SELECT
        id,
        case when solved_at_pst is not null then {{ cxo_business_mins_between__2('created_at_pst', 'solved_at_pst') }} * 1.0, 
        {{ date_formatting('date_five9') }}, 
        {{ dbt_utils.generate_surrogate_key(['vigilancedecisionid', 'createddate']) }} AS unique_id,
        {{ dbt_utils.generate_surrogate_key('id', 'timestamp') }} AS other_unique id, 
        {{ fico_extract('fico_range') }} as vantage_score,
        isactive,
        cast(activity_date as {{ dbt.type_timestamp() }}) as activity_timestamp,
        name,
        createddate,
        createdby,
        updateddate,
        updatedby,
        addresscity,
        orgvcardstatus,
        vendorvcardstatus,
        orgvcardoptoutreason,
        vstatusupdatesource,
        intlpaymenttype,
        processorvcardstatus,
        ,{{ mtn_time_conve

In [71]:
def process_sql_files(directory: str, processor: DbtJinjaProcessor, error_log_path: str) -> None:
    """
    Use a DbtJinjaProcessor-like 'processor' to process all .sql files
    in the given directory and its subdirectories.
    The processed SQL will overwrite the original files.
    Any errors will be logged to error_log_path.
    """
    # Find all .sql files in the directory (recursively)
    sql_files = glob.glob(os.path.join(directory, '**', '*.sql'), recursive=True)
    total_files = len(sql_files)
    
    print(f"Total .sql files to process (for Jinja): {total_files}")
    
    processed_files = []
    failed_files = []

    for filepath in sql_files:
        try:
            with open(filepath, 'r') as file:
                content = file.read()
            
            # Process the template
            processed_content = processor.process_template_str(content)
            
            # Overwrite the original file with the processed content
            with open(filepath, 'w') as file:
                file.write(processed_content)
            
            processed_files.append(filepath)
            print(f"Processed (Jinja) {filepath} successfully.")
        
        except Exception as e:
            failed_files.append(filepath)
            error_message = (
                f"Failed to process (Jinja) {filepath}: {e}\n{traceback.format_exc()}"
            )
            print(error_message)
            with open(error_log_path, 'a') as error_log_file:
                error_log_file.write(error_message + "\n")

    # Print summary
    print("\nJinja Processing Summary:")
    print(f"Successfully processed files: {len(processed_files)}")
    print(f"Failed files: {len(failed_files)}")

    if failed_files:
        print("Failed Jinja files:")
        for failed_file in failed_files:
            print(failed_file)

In [72]:
processor = DbtJinjaProcessor(excluded_config_args=["dist", "sort", "unique_id", "distkey", "dist_key", "sort_key", "sortkey"])
error_log_path = 'error_log_jinja.txt'

process_sql_files('/Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/', processor, error_log_path)

Total .sql files to process (for Jinja): 2662
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__unified_business_mapping_sfdc.sql successfully.
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__enriched_apar_unify_prep.sql successfully.
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__unified_business_keys.sql successfully.
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__unified_business_exclusion.sql successfully.
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__prob_id_final.sql successfully.
Processed (Jinja) /Users/dvaliaiev/PycharmProjects/pythonProject1/edwh-dbt/models/staging/treasure_data_cdp/stg_cdp__enriched_se_unify_prep.sql successfully.
Pro